In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch

from tsfm_lens.chronos.circuitlens import CircuitLensChronos
from tsfm_lens.utils import (
    apply_custom_style,
    collect_attributions,
    compute_logit_metrics,
    make_ensemble_from_arrow_dir,
    plot_output_logits,
    plot_output_logits_across_layers,
    set_seed,
)

In [ ]:
apply_custom_style("../../config/plotting.yaml")

In [ ]:
DEFAULT_COLORS = list(plt.get_cmap("tab20").colors)  # type: ignore
print(f"{len(DEFAULT_COLORS)} colors")

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
WORK_DIR = os.getenv("WORK", "/work")
DATA_DIR = os.path.join(WORK_DIR, "data")
plot_save_dir = os.path.join("../figs", "logit_attribution")
os.makedirs(plot_save_dir, exist_ok=True)


### Load Pipeline

In [ ]:
pipeline = CircuitLensChronos.from_pretrained("amazon/chronos-t5-base", device_map=device, torch_dtype=torch.bfloat16)
num_layers = pipeline.model.model.config.num_decoder_layers
num_heads = pipeline.model.model.config.num_heads

print(f"num_layers: {num_layers}, num_heads: {num_heads}")

### Load Data

In [ ]:
split_name = "base40"
split_dir = os.path.join(DATA_DIR, split_name)

system_name = "Thomas"
subsample_interval = 1

ensemble = make_ensemble_from_arrow_dir(split_dir, dyst_names_lst=[system_name], num_samples=1)

In [ ]:
context_length = 512
context_start_time = 2000
context_end_time = context_start_time + context_length

In [ ]:
# Prepare input series
sample_idx = 0
selected_dim = 0
# get sample sample_idx of system_name
dyst_coords = torch.tensor(ensemble[system_name][sample_idx, selected_dim, :]).unsqueeze(0)
print(dyst_coords.shape)
dyst_coords = dyst_coords[:, ::subsample_interval]
context = dyst_coords[:, context_start_time:context_end_time]

print(context.shape)

In [ ]:
pipeline.model.model

In [ ]:
prediction_length = 512

In [ ]:
future_vals = dyst_coords[:, context_end_time : context_end_time + prediction_length]
print(f"future_vals shape: {future_vals.shape}")

In [ ]:
pipeline.__class__.__name__

### Add Hooks

Remove Hooks and Attribution Inputs/Outputs 

### Ablations

In [ ]:
# To be extra safe, after we load the pipeline, we remove all hooks and also clear all attribution inputs and outputs
pipeline.remove_all_hooks()  # NOTE: check if this also removes attribution inputs and outputs

# NOTE: this is the top 30% of highest entropy heads (excluding all the heads in layer 0). Note that it preserves the parroting
# Specifically, these heads were identified from a context-parroting window of Lorenz system (see notebooks/kernel_analogy/head_sharpness.ipynb)
heads_to_ablate = [
    # (0, 4),  # layer 0
    # (10, 11),
    # (9, 5),
    # (10, 5),
    # (0, 11),  # layer 0
    # (5, 8),
    # (8, 6),
    # (10, 8),
    # (5, 2),
    # (10, 1),
    # (10, 4),
    # (7, 2),
    # (10, 10),
    # (9, 1),
    # (0, 10),  # layer 0
    # (2, 8),
    # (10, 9),
    # (9, 0),
    # # (0, 1),
    # (10, 7),
    # (1, 1),
    # (6, 10),
    # (10, 0),
    # (2, 0),
    # (11, 6),
    # (8, 3),
    # (9, 4),
    # (2, 2),
    # (5, 1),
    # # (0, 6),
    # (4, 6),
    # (2, 4),
    # (0, 7),
    # (7, 10),
    # (9, 3),
    # (4, 3),
    # (6, 2),
    # (11, 5),
    # # (0, 5),
    # (7, 1),
    # (7, 3),
    # (6, 6),
    # (3, 7),
    # (6, 8),
    # # (0, 0),
    # (11, 11),
    # (7, 8),
    # (10, 2),
    # (11, 3),
    # (7, 9),
    # (1, 3),
]


######### Attention Head ablations #########
pipeline.add_head_ablation_hooks(
    heads_to_ablate,
    attention_type="ca",
)
pipeline.add_head_ablation_hooks(
    heads_to_ablate,
    attention_type="sa",
)

######### MLP ablations #########
# pipeline.add_mlp_ablation_hooks(layers_to_ablate, ablation_method="zero")
# pipeline.add_mlp_ablation_hooks([1, 2], ablation_method="zero")

In [ ]:
n_heads_ablated = len(heads_to_ablate)
print(f"n_heads_ablated: {n_heads_ablated}")

In [ ]:
# print number of heads ablated per layer
for layer in list(range(pipeline.num_layers)):
    heads_in_layer = [config[1] for config in heads_to_ablate if config[0] == layer]
    print(f"Layer {layer}: {len(heads_in_layer)} heads")

In [ ]:
layers_without_ca_heads = list(pipeline.ca_head_ablation_handles.keys())
layers_without_sa_heads = list(pipeline.sa_head_ablation_handles.keys())
layers_without_mlps = list(pipeline.mlp_ablation_handles.keys())

assert layers_without_ca_heads == layers_without_sa_heads, "CA and SA ablations must be identical"
layers_without_heads = layers_without_ca_heads

ablations_summary_str = f"ablate_{n_heads_ablated}_heads"

if layers_without_heads and layers_without_mlps:
    if layers_without_heads != layers_without_mlps:
        # raise NotImplementedError("Zero-ablation of heads and MLPs in different layers is messier, save for later")
        pass
    ablations_summary_str_title = f"Zero-Ablation: Heads and MLPs in Layers {layers_without_heads}"
    ablations_summary_str_suffix = f"za_heads_mlps_layers_{'-'.join(map(str, layers_without_heads))}"
elif layers_without_heads:
    ablations_summary_str_title = f"Zero-Ablation: Heads in Layers {layers_without_heads}"
    ablations_summary_str_suffix = f"za_heads_layers_{'-'.join(map(str, layers_without_heads))}"
elif layers_without_mlps:
    ablations_summary_str_title = f"Zero-Ablation: MLPs in Layers {layers_without_mlps}"
    ablations_summary_str_suffix = f"za_mlps_layers_{'-'.join(map(str, layers_without_mlps))}"
else:
    ablations_summary_str_suffix = ablations_summary_str_title = None

if ablations_summary_str_suffix is not None:
    ablations_summary_str += "_" + ablations_summary_str_suffix

print(ablations_summary_str)
print(ablations_summary_str_title)


### Attribution

In [ ]:
pipeline.reset_attribution_inputs_and_outputs()

# Add hooks for head attribution
# print("Adding SA head attribution hooks")
# pipeline.add_head_attribution_hooks(
#     [(i, j) for i in range(num_layers) for j in range(num_heads)],
#     attention_type="sa",
# )
print("Adding CA head attribution hooks")
pipeline.add_head_attribution_hooks(
    [(i, j) for i in range(num_layers) for j in range(num_heads)],
    attention_type="ca",
)
print("Adding MLP attribution hooks")
pipeline.add_mlp_attribution_hooks([i for i in range(num_layers)])

print("Adding read stream hooks")
pipeline.add_read_stream_hooks([i for i in range(num_layers)])

print("Adding SA attn output attribution hooks")
pipeline.add_attn_attribution_hooks([i for i in range(num_layers)], attention_type="sa")
print("Adding CA attn output attribution hooks")
pipeline.add_attn_attribution_hooks([i for i in range(num_layers)], attention_type="ca")

In [ ]:
print([k for (k, v) in pipeline.__dict__.items() if v])

### Predict and Get Outputs

In [ ]:
rseed = 123
set_seed(rseed)

In [ ]:
num_samples = 10
do_sample = True if num_samples > 1 else False

In [ ]:
pred, encdec_out = pipeline.predict_with_full_outputs(  # type: ignore
    context=context,
    prediction_length=prediction_length,
    num_samples=num_samples,
    do_sample=do_sample,  # greedy decoding if num_samples == 1
    use_cache=True,
    return_dict_in_generate=True,
    output_scores=True,
    limit_prediction_length=False,
)

In [ ]:
do_sample

### Plot Predictions

In [ ]:
fig, ax = plt.subplots(figsize=(6, 2))

# Prepare context and predictions
ctx = context.squeeze().cpu().numpy() if hasattr(context, "cpu") else context.squeeze()
future_timesteps = np.arange(context_length, context_length + prediction_length)
future_vals_np = future_vals.squeeze()
preds = pred.squeeze()
if preds.ndim == 1:
    preds = preds[None, :]
n, L = preds.shape

# Plot context and ground truth
ax.plot(np.arange(len(ctx)), ctx, color="black", linewidth=1, label="Context")
ax.plot(
    future_timesteps,
    future_vals_np,
    color="black",
    linewidth=1,
    linestyle="--",
    label="Ground Truth",
)

# Plot predictions
for i in range(n):
    ax.plot(
        np.arange(len(ctx), len(ctx) + L),
        preds[i],
        color=DEFAULT_COLORS[i],
        linewidth=1,
        alpha=0.5,
    )

# plot median
plt.plot(
    future_timesteps,
    np.median(preds, axis=0),
    color="tab:blue",
    linewidth=2,
    label="Median",
)

ax.set_xlabel("Timestep", fontweight="bold")


save_fname = f"predictions_{system_name}_dim{selected_dim}_start{context_start_time}"
save_path = os.path.join(plot_save_dir, f"{save_fname}.pdf")
if ablations_summary_str_title:
    print(ablations_summary_str_title)
    # ax.set_title(ablations_summary_str_title, fontweight="bold")
    save_path = os.path.join(plot_save_dir, "ablations", f"{ablations_summary_str}_{save_fname}.pdf")

print(f"Saving to {save_path}")
os.makedirs(os.path.dirname(save_path), exist_ok=True)
fig.tight_layout()
fig.savefig(save_path, bbox_inches="tight")
plt.show()

In [ ]:
ablations_summary_str

## Visualize

In [ ]:
len(encdec_out)

In [ ]:
encdec_out[0].__dict__.keys()

In [ ]:
encdec_out[0].sequences.shape  # type: ignore

In [ ]:
all_predicted_token_ids = np.concatenate(
    [out.sequences[:, 1:].cpu().numpy() for out in encdec_out],  # type: ignore
    axis=1,
)
all_predicted_token_ids.shape

In [ ]:
head_outputs = {
    i: [collect_attributions(pipeline.ca_head_attribution_outputs[i][j]) for j in range(num_heads)]
    for i in range(num_layers)
}

mlp_outputs = {i: collect_attributions(pipeline.mlp_attribution_outputs[i]) for i in range(num_layers)}

read_stream_outputs = {i: collect_attributions(pipeline.read_stream_outputs[i]) for i in range(num_layers)}

In [ ]:
read_stream_outputs[0].shape

In [ ]:
for name, outs in [
    ("head", head_outputs),
    ("mlp", mlp_outputs),
    ("read_stream", read_stream_outputs),
]:
    for out in outs.values():
        shape = out[0].shape if name == "head" else out.shape
        assert shape[0] == num_samples, f"{name} shape[0] = {shape[0]} != num_samples = {num_samples}"


In [ ]:
# Concise plot: context, head outputs (layer 0), and actual output
_, _, scale = pipeline.tokenizer.context_input_transform(context)

context_tokens = pipeline.tokenizer._input_transform(context, scale)[0]

In [ ]:
context_tokens.shape

In [ ]:
all_predicted_token_ids.shape

In [ ]:
future_timesteps.shape

In [ ]:
selected_parallel_sample_idx = 0
selected_layer_idx = 11

plt.figure(figsize=(12, 6))
plt.plot(
    range(len(ctx)),
    context_tokens.squeeze(),
    "--",
    color="gray",
    label="Context",
)
for j in range(num_heads):
    tokens = pipeline.sample_tokens(head_outputs[selected_layer_idx][j][selected_parallel_sample_idx]).cpu().numpy()
    plt.plot(
        future_timesteps,
        tokens,
        label=f"head {j}",
        color=DEFAULT_COLORS[j],
        linewidth=1,
        zorder=1,
    )
plt.plot(
    future_timesteps,
    all_predicted_token_ids[selected_parallel_sample_idx],
    color="black",
    linewidth=2,
    label="Prediction",
    zorder=2,
)
plt.axvline(x=context_length, color="r", linestyle="-", alpha=0.3)
plt.legend(loc="lower left", ncol=2, frameon=True, fontsize=12)
plt.title(f"Head Outputs for Layer {selected_layer_idx}", fontweight="bold")
plt.xlabel("Timestep", fontweight="bold")
plt.ylabel("Token ID", fontweight="bold")
plt.tight_layout()
plt.savefig(
    os.path.join(plot_save_dir, f"head_outputs_layer{selected_layer_idx}_{system_name}.pdf"),
    bbox_inches="tight",
)
plt.show()

In [ ]:
mlp_outputs[0].shape

In [ ]:
def output_transform_tokens_using_context(tokens, context):
    _, _, scale = pipeline.tokenizer.context_input_transform(context)
    scale = scale.to(torch.float32)
    print(f"scale: {scale}")
    return pipeline.tokenizer.output_transform(torch.tensor(tokens).to(scale.device), scale)

In [ ]:
selected_layer_idx = 0

output_type = "read_stream"
keep_as_tokens = False

if output_type == "mlp":
    output = mlp_outputs
    output_title_name = "MLP"
elif output_type == "read_stream":
    output = read_stream_outputs
    output_title_name = "Read Stream"
else:
    raise ValueError(f"Invalid output type: {output_type}")

token_preds_from_attribution_output = np.stack(
    [pipeline.sample_tokens(output[selected_layer_idx][j]).cpu().numpy() for j in range(num_samples)]
)

preds_from_attribution_output = token_preds_from_attribution_output
if not keep_as_tokens:
    preds_from_attribution_output = output_transform_tokens_using_context(token_preds_from_attribution_output, context)
print(preds_from_attribution_output.shape)
# NOTE: our plotting functionality expects a 2D array of shape (num_samples, prediction_length) (no batch dimension)
preds_from_attribution_output = preds_from_attribution_output.squeeze()

plt.figure(figsize=(12, 6))
plt.plot(
    np.arange(len(ctx)),
    context_tokens.squeeze() if keep_as_tokens else context.squeeze(),
    "--",
    color="gray",
    label="context",
)
for j, tokens in enumerate(preds_from_attribution_output):
    plt.plot(
        future_timesteps,
        tokens,
        label=f"Sample {j}",
        color=DEFAULT_COLORS[j],
        linewidth=1,
    )
plt.plot(
    future_timesteps,
    np.median(preds_from_attribution_output, axis=0),
    label=f"{output_title_name} Median",
    color="tab:blue",
    linewidth=2,
)
plt.plot(
    future_timesteps,
    np.median(all_predicted_token_ids if keep_as_tokens else preds, axis=0),
    color="black",
    linewidth=2,
    label="Median Prediction",
)
plt.axvline(x=context_length, color="r", linestyle="-", alpha=0.3)
plt.legend(loc="upper left", ncol=1, frameon=True, fontsize=12)
plt.title(f"{output_title_name} Outputs for Layer {selected_layer_idx}", fontweight="bold")
plt.xlabel("Timestep", fontweight="bold")
plt.ylabel("Token ID" if keep_as_tokens else "x", fontweight="bold")
plt.tight_layout()
plt.savefig(
    os.path.join(
        plot_save_dir,
        f"{output_type}_outputs_layer{selected_layer_idx}_{system_name}.pdf",
    ),
    bbox_inches="tight",
)
plt.show()

In [ ]:
preds[0, :20]
## Residual Stream

In [ ]:
preds_from_attribution_output[0, -20:]

In [ ]:
preds_from_attribution_output[0, -20:]

In [ ]:
mae = torch.mean(torch.abs(preds - preds_from_attribution_output))
print(f"MAE: {mae}")


In [ ]:
def smape(a, b):
    # sMAPE = 100 * mean(2 * |a - b| / (|a| + |b|))
    denominator = torch.abs(a) + torch.abs(b)
    # To avoid division by zero, add a small epsilon
    epsilon = 1e-8
    smape_val = 100 * torch.mean(2.0 * torch.abs(a - b) / (denominator + epsilon))
    return smape_val


smape_val = smape(preds, preds_from_attribution_output) / 2
print(f"sMAPE: {smape_val}")

In [ ]:
plt.plot(preds[2])
plt.plot(preds_from_attribution_output[2])

## Residual Stream

In [ ]:
plot_output_logits_across_layers(
    pipeline,
    read_stream_outputs,
    prediction_length,
    output_type="Read Stream",
    token_id_range=(1600, 2700),
    plot_save_path=os.path.join(plot_save_dir, f"read_stream_across_layers_{system_name}.pdf"),
)

In [ ]:
layer_idx = 0
save_path = os.path.join(plot_save_dir, f"read_stream_layer-{layer_idx}_{system_name}.pdf")
plot_output_logits(
    pipeline,
    read_stream_outputs,
    prediction_length,
    plot_save_path=save_path,
    selected_parallel_sample_idx=0,
    selected_layer_idx=layer_idx,
    token_id_range=(1600, 2600),
    figsize=(6, 6),
    show_colorbar=False,
)
print(f"plot saved to {save_path}")

In [ ]:
torch.argmax(pipeline.unembed_residual(read_stream_outputs[0][0])[1])

## CA Heads

In [ ]:
layer_idx = 0
plot_output_logits_across_layers(
    pipeline,
    head_outputs[layer_idx],
    prediction_length,
    output_type="Head",
    title=f"CA Head Outputs for Layer {layer_idx}",
    token_id_range=(1200, 3000),
    plot_save_path=os.path.join(plot_save_dir, f"head_outputs_across_layers_{system_name}_layer{layer_idx}.pdf"),
)

## MLPs

In [ ]:
plot_output_logits_across_layers(
    pipeline,
    mlp_outputs,
    prediction_length,
    output_type="MLP",
    token_id_range=(1200, 3000),
    plot_save_path=os.path.join(plot_save_dir, f"mlp_across_layers_{system_name}.pdf"),
)

In [ ]:
layer_idx = 11
plot_output_logits(
    pipeline,
    mlp_outputs,
    prediction_length,
    plot_save_path=os.path.join(plot_save_dir, f"mlp_layer-{layer_idx}_{system_name}.pdf"),
    selected_parallel_sample_idx=0,
    selected_layer_idx=layer_idx,
    token_id_range=(1200, 3000),
    figsize=(6, 6),
)

### Compare MLP vs CA and SA Attributions for Ablated layer

In [ ]:
# NOTE:  hard coded for layer 4 ablation
ca_outputs = {i: collect_attributions(pipeline.ca_attribution_outputs[i]) for i in range(num_layers)}

sa_outputs = {i: collect_attributions(pipeline.sa_attribution_outputs[i]) for i in range(num_layers)}

In [ ]:
layer_idx = 4
plot_output_logits(
    pipeline,
    ca_outputs,
    prediction_length,
    plot_save_path=None,
    selected_parallel_sample_idx=0,
    selected_layer_idx=layer_idx,
    token_id_range=(1200, 3000),
    figsize=(6, 6),
)

In [ ]:
len(ca_outputs)

In [ ]:
plot_output_logits_across_layers(
    pipeline,
    ca_outputs,
    prediction_length,
    output_type="CA Attribution",
    token_id_range=(1200, 3000),
    plot_save_path=os.path.join(plot_save_dir, f"ca_attribution_across_layers_{system_name}.pdf"),
)

In [ ]:
torch.argmax(pipeline.unembed_residual(mlp_outputs[0][0])[1])

## Measuring Lines of Thought

In [ ]:
read_stream_outputs.keys()

In [ ]:
# shape: (num_samples, prediction_length, d_model)
read_stream_outputs[0].shape

In [ ]:
sample_idx = 5
layer_idx = 0
read_stream_unembed = pipeline.unembed_residual(read_stream_outputs[layer_idx][sample_idx])
# shape: (prediction_length, vocab_size)
print(read_stream_unembed.shape)

In [ ]:
logits = read_stream_unembed.detach().cpu().float().numpy()

In [ ]:
logits.shape

In [ ]:
line_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

prediction_length, vocab_size = logits.shape
token_prob_threshold = 0.01
top_k_tokens_to_consider = 10
entropy, effective_vocab, top_k_entropy, peak_counts = compute_logit_metrics(
    logits,
    threshold=token_prob_threshold,
    top_k=top_k_tokens_to_consider,
)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes = axes.flatten()

timesteps = np.arange(prediction_length)

titles = [
    "Entropy",
    f"Effective Vocab Size (p>{token_prob_threshold})",
    f"Top-{top_k_tokens_to_consider} Entropy",
]
ylabels = ["Entropy", "Count", "Entropy"]
data = [entropy, effective_vocab, top_k_entropy]

for i, (ax, title, ylabel, y) in enumerate(zip(axes, titles, ylabels, data)):
    ax.plot(timesteps, y, linewidth=2, color=line_colors[i])
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Timestep")
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)

plt.tight_layout()

In [ ]:
entropy.shape

In [ ]:
top_k_entropy.mean()


In [ ]:
num_layers

In [ ]:
outputs_dict = {
    "read_stream": read_stream_outputs,
    "mlp": mlp_outputs,
    "ca": ca_outputs,
    "sa": sa_outputs,
}

In [ ]:
output_name = "read_stream"
outputs = outputs_dict[output_name]

sample_idx = 4
result_per_layer = {}
for layer_idx in range(num_layers):
    logits = pipeline.unembed_residual(outputs[layer_idx][sample_idx]).detach().cpu().float().numpy()
    entropy, effective_vocab, top_k_entropy, peak_counts = compute_logit_metrics(
        logits,
        threshold=token_prob_threshold,
        top_k=top_k_tokens_to_consider,
    )
    avg_entropy = entropy.mean()
    avg_effective_vocab = effective_vocab.mean()
    avg_top_k_entropy = top_k_entropy.mean()
    result_per_layer[layer_idx] = {
        "entropy": avg_entropy,
        "effective_vocab": avg_effective_vocab,
        "top_k_entropy": avg_top_k_entropy,
    }

In [ ]:
quantity_name = "entropy"
result_per_layer_data = np.stack(
    [result_per_layer[layer_idx][quantity_name] for layer_idx in range(num_layers)],
    axis=0,
)
print(result_per_layer_data.shape)


In [ ]:
save_fig = False

quantity_title_name = quantity_name.replace("_", " ").title()

if quantity_name == "effective_vocab":
    quantity_title_name = f"Effective Vocab Size $(\\mathrm{{p}} > {token_prob_threshold})$"
elif quantity_name == "top_k_entropy":
    quantity_title_name = f"Entropy of Top {top_k_tokens_to_consider} Tokens"

plt.figure(figsize=(4, 4))
plt.plot(result_per_layer_data, marker="o", markersize=4, linestyle="-")
plt.xlabel("Layer Index", fontweight="bold")
plt.ylabel(quantity_title_name, fontweight="bold")
# plt.title(f"{quantity_title_name}", fontweight="bold")
plt.tight_layout()
if save_fig:
    plt.savefig(
        os.path.join(plot_save_dir, f"{quantity_name}_per_layer_{system_name}.pdf"),
        bbox_inches="tight",
    )

In [ ]:
layer_idx = 6
sample_idx = 0
logits = pipeline.unembed_residual(read_stream_outputs[layer_idx][sample_idx]).detach().cpu().float().numpy()
print(logits.shape)

In [ ]:
import torch.nn.functional as F

# probs shape (T, V)
probs = F.softmax(torch.from_numpy(logits), dim=-1).numpy()
print(probs.shape)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4, 4))

im = ax.imshow(
    probs[:prediction_length, :].T,
    aspect="auto",
    cmap="Blues",
    vmin=0,
    vmax=0.1,
)
ax.set_ylim(3000, 1000)
ax.invert_yaxis()

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Probability", fontsize=12, weight="bold")

plt.show()

In [ ]:
# logits_only_positive = np.where(logits < 0, 0, logits)

In [ ]:
np.sort(probs[:10], axis=1)[:, -2:]

In [ ]:
from tsfm_lens.utils import peak_count_prominence

In [ ]:
import time

start_time = time.time()
peak_counts, peak_indices = peak_count_prominence(probs, height=0.1, min_distance=2, ensure_pmf=False)
end_time = time.time()
print(f"Time taken: {end_time - start_time} seconds")
print(f"Max number of peaks: {int(np.max(peak_counts))}")
print(f"Mean number of peaks: {float(np.mean(peak_counts))}")

In [ ]:
np.argsort(peak_counts[:50])[:3]

In [ ]:
output_name = "read_stream"
outputs = outputs_dict[output_name]

sample_idx = 1
peak_counts_per_layer = {}
for layer_idx in range(num_layers):
    logits = pipeline.unembed_residual(outputs[layer_idx][sample_idx]).detach().cpu().float().numpy()

    # probs shape (T, V)
    probs = F.softmax(torch.from_numpy(logits), dim=-1).numpy()

    # logits_max = np.max(logits, axis=-1, keepdims=True)
    # exp_logits = np.exp(logits - logits_max)
    # probs = exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)

    peak_counts, _ = peak_count_prominence(probs, height=0.02, min_distance=2, ensure_pmf=False)
    peak_counts_per_layer[layer_idx] = float(np.mean(peak_counts))


In [ ]:
layer_indices = list(peak_counts_per_layer.keys())
peak_counts = list(peak_counts_per_layer.values())


In [ ]:
plt.figure(figsize=(4, 4))
plt.plot(layer_indices, peak_counts, marker="o", markersize=4, linestyle="-")
plt.xlabel("Layer Index", fontweight="bold")
plt.ylabel("Mean Number of Peaks", fontweight="bold")
plt.tight_layout()
plt.show()
